# 03 - Model Training

**Goal**: Fit a Random Forest classifier on the cleaned data from notebook 02, save it to `../models/detector.pkl`, and record training metrics in `metadata.json`.

**What this notebook does**:
1. Load `cleaned.parquet` + `feature_names.json` (our feature contract)
2. Stratified train / val / test split (70 / 15 / 15)
3. Fit `RandomForestClassifier(n_estimators=100, class_weight='balanced')`
4. Quick validation-set sanity check
5. Save `detector.pkl` + update `metadata.json`

**Not doing**: hyperparameter tuning. RF defaults work well on CICIDS2017 and the backend-architect review explicitly cut grid search from scope. If notebook 04's evaluation shows bad performance on a specific class, we'll revisit here.

**Not saving**: a `StandardScaler`. Random Forests are tree-based and scale-invariant, feature scaling does nothing for them. Shipping a scaler would be dead weight at runtime.

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

DATA_DIR   = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH  = DATA_DIR / 'cleaned.parquet'
FEATURES_PATH = MODELS_DIR / 'feature_names.json'
METADATA_PATH = MODELS_DIR / 'metadata.json'
MODEL_PATH    = MODELS_DIR / 'detector.pkl'

RANDOM_STATE = 42

## 1. Load data + verify the feature contract

`feature_names.json` is our single source of truth for feature order. If the parquet and JSON disagree, we bail out loudly rather than silently train on mismatched data.

In [2]:
df       = pd.read_parquet(PARQUET_PATH)
FEATURES = json.loads(FEATURES_PATH.read_text())
metadata = json.loads(METADATA_PATH.read_text())
CLASS_NAMES = metadata['class_names']

# Contract check: every feature in the JSON must exist in the parquet, in exact snake_case spelling.
missing = [f for f in FEATURES if f not in df.columns]
assert not missing, f'Contract violation: {missing} missing from parquet'

X = df[FEATURES].to_numpy(dtype=np.float64)
y = df['label'].to_numpy(dtype=int)

print(f'Samples:     {len(df):,}')
print(f'Features:    {X.shape[1]}  (order from feature_names.json)')
print(f'Classes:     {CLASS_NAMES}')
print(f'Class dist:  {dict(zip(*np.unique(y, return_counts=True)))}')

Samples:     437,207
Features:    15  (order from feature_names.json)
Classes:     ['benign', 'ddos', 'portscan']
Class dist:  {0: 218372, 1: 128016, 2: 90819}


## 2. Stratified train / val / test split (70 / 15 / 15)

We split in two steps:
1. First carve off 15% as test (never touched until notebook 04).
2. From the remaining 85%, split 15/85 → val / train, so final ratios are 70 / 15 / 15.

`stratify=y` ensures each split preserves the class ratios (critical when PortScan is the smallest class).

In [3]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE,
)

# 0.15 / 0.85 ≈ 0.1765 → 15 percentage points out of the remaining 85.
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85, stratify=y_trainval, random_state=RANDOM_STATE,
)

def split_summary(name: str, y_split: np.ndarray) -> dict:
    n = len(y_split)
    row = {'split': name, 'n': n}
    for class_idx, cname in enumerate(CLASS_NAMES):
        row[cname] = f'{(y_split == class_idx).sum():,}'
    row['share'] = f'{n / len(y):.1%}'
    return row

pd.DataFrame([
    split_summary('train', y_train),
    split_summary('val',   y_val),
    split_summary('test',  y_test),
])

,split,n,benign,ddos,portscan,share
0,train,306044,"152,860","89,611","63,573",70.0%
1,val,65581,"32,756","19,202","13,623",15.0%
2,test,65582,"32,756","19,203","13,623",15.0%


## 3. Train the Random Forest

Why these settings:
- `n_estimators=100` - plenty for 15 features; more trees → slower inference at runtime
- `class_weight='balanced'` - compensates for the 50/29/21 class imbalance without downsampling
- `n_jobs=-1` - use all cores during training (inference is single-threaded anyway)
- `random_state=42` - reproducibility for the report

Training on ~305K rows × 15 features × 100 trees

In [4]:
clf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

t0 = time.time()
clf.fit(X_train, y_train)
train_time_s = time.time() - t0

print(f'Trained in {train_time_s:.1f}s')
print(f'Model size: {sum(t.tree_.node_count for t in clf.estimators_):,} total nodes across {len(clf.estimators_)} trees')

Trained in 2.9s
Model size: 56,726 total nodes across 100 trees


## 4. Validation-set sanity check

We score on the **validation set only**, not test. The test set is reserved for the final evaluation in notebook 04 so that our severity-threshold calibration doesn't contaminate the reported test metrics.

In [5]:
y_val_pred = clf.predict(X_val)

val_accuracy = float(accuracy_score(y_val, y_val_pred))
val_macro_f1 = float(f1_score(y_val, y_val_pred, average='macro'))
val_per_class_f1 = f1_score(y_val, y_val_pred, average=None, labels=range(len(CLASS_NAMES)))

print(f'Validation accuracy : {val_accuracy:.4f}')
print(f'Validation macro F1 : {val_macro_f1:.4f}')
print()
print(classification_report(y_val, y_val_pred, target_names=CLASS_NAMES, digits=4))

Validation accuracy : 0.9996
Validation macro F1 : 0.9997

              precision    recall  f1-score   support

      benign     0.9996    0.9997    0.9996     32756
        ddos     0.9995    0.9995    0.9995     19202
    portscan     0.9999    0.9997    0.9998     13623

    accuracy                         0.9996     65581
   macro avg     0.9997    0.9996    0.9997     65581
weighted avg     0.9996    0.9996    0.9996     65581



### Interpreting the validation report

What we want to see (roughly):
- **Macro F1 ≥ 0.95** - model is learning the three classes well
- **Recall ≥ 0.95 on both attack classes** - we don't want to miss attacks
- **Precision ≥ 0.95 on both attack classes** - we don't want to flood ops with false positives

If any class is noticeably below 0.90, that's a signal. Most likely culprit: a feature that *should* discriminate that class isn't in the selected set. Revisit notebook 02 feature picks if that happens.

## 5. Preview inference output

At runtime `detector.py` will call `predict_proba` to get per-class probabilities, then apply severity thresholds. Quick preview so you know what the runtime sees.

In [6]:
# Grab one example of each class from the val set and show predict_proba output
for class_idx, cname in enumerate(CLASS_NAMES):
    idx = np.where(y_val == class_idx)[0][0]
    probs = clf.predict_proba(X_val[idx:idx+1])[0]
    pred_class = CLASS_NAMES[int(np.argmax(probs))]
    print(f'True: {cname:<9} Predicted: {pred_class:<9}  probs = {dict(zip(CLASS_NAMES, np.round(probs, 3)))}')

True: benign    Predicted: benign     probs = {'benign': 1.0, 'ddos': 0.0, 'portscan': 0.0}
True: ddos      Predicted: ddos       probs = {'benign': 0.0, 'ddos': 1.0, 'portscan': 0.0}
True: portscan  Predicted: portscan   probs = {'benign': 0.0, 'ddos': 0.0, 'portscan': 1.0}


## 6. Save the trained model + update metadata

We use `joblib.dump` (scikit-learn's recommended serializer, handles large numpy arrays better than `pickle`).

In [7]:
joblib.dump(clf, MODEL_PATH, compress=3)
model_size_mb = MODEL_PATH.stat().st_size / 1024 / 1024

# Extend metadata with training info (preserving feature_names / class_names from notebook 02)
metadata.update({
    'model': {
        'type': 'RandomForestClassifier',
        'n_estimators': int(clf.n_estimators),
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE,
    },
    'splits': {
        'train': int(len(y_train)),
        'val':   int(len(y_val)),
        'test':  int(len(y_test)),
        'strategy': 'stratified 70/15/15',
    },
    'training_metrics': {
        'train_time_seconds': round(train_time_s, 2),
        'val_accuracy':  round(val_accuracy, 4),
        'val_macro_f1':  round(val_macro_f1, 4),
        'val_per_class_f1': {
            CLASS_NAMES[i]: round(float(val_per_class_f1[i]), 4)
            for i in range(len(CLASS_NAMES))
        },
    },
    'trained_at': datetime.now(timezone.utc).isoformat(),
})

METADATA_PATH.write_text(json.dumps(metadata, indent=2))

print(f'Saved: {MODEL_PATH}  ({model_size_mb:.1f} MB)')
print(f'Saved: {METADATA_PATH}')
print()
print('Updated metadata.json:')
print(json.dumps({k: metadata[k] for k in ['model', 'splits', 'training_metrics']}, indent=2))

Saved: ..\models\detector.pkl  (1.5 MB)
Saved: ..\models\metadata.json

Updated metadata.json:
{
  "model": {
    "type": "RandomForestClassifier",
    "n_estimators": 100,
    "class_weight": "balanced",
    "random_state": 42
  },
  "splits": {
    "train": 306044,
    "val": 65581,
    "test": 65582,
    "strategy": "stratified 70/15/15"
  },
  "training_metrics": {
    "train_time_seconds": 2.92,
    "val_accuracy": 0.9996,
    "val_macro_f1": 0.9997,
    "val_per_class_f1": {
      "benign": 0.9996,
      "ddos": 0.9995,
      "portscan": 0.9998
    }
  }
}


## Summary + next

- `models/detector.pkl` is trained and saved
- `metadata.json` now carries training metrics for the report
- Test set (~65K rows) untouched, reserved for final evaluation

**Next**: `04_evaluation.ipynb` - confusion matrix, ROC curves, feature-importance plot (all saved to `report/figures/` for the PDF), and **threshold calibration** to set the warn/critical cutoffs from data rather than guessing.